# Efficient LLM Serving at Scale with Unified Caching (vLLM + LMCache)

**The idea:** vLLM's prefix cache lives in GPU memory. When many long requests overflow that GPU pool, their KV gets **evicted and recomputed** on every repeat. LMCache adds a CPU-DRAM tier: evicted KV is **reloaded from host memory** instead of recomputed — cutting time-to-first-token (TTFT).

**We run the same overflow workload twice (cold then warm) under two servers:**

| Server | Warm-pass TTFT |
|---|---|
| vLLM prefix cache only | stays high — working set exceeds GPU pool, recomputed every time |
| vLLM prefix cache **+ LMCache** | **drops** — evicted KV reloaded from CPU |

Workload: `long-doc-qa` (long documents, ISL 24K, decode-bound) sized to **overflow the GPU KV pool**. Model: `google/gemma-4-31B-it` (a sliding-window **hybrid** model). Runs **inside `vllm/vllm-openai-rocm:v0.23.0`** on one AMD Instinct GPU (MI300X/MI325X = `gfx942`, MI350X/MI355X = `gfx950`), with **LMCache built from source for ROCm**.

> ⚠️ **Status:** the install, native-transfer, and server config are verified on ROCm. The exact overflow operating point (`--gpu-memory-utilization`, `OVERFLOW`, `--ldqa-*`) is calibrated from the validated hybrid-benchmarking methodology and may need one tuning pass on your hardware to make the warm-pass drop clear.

## 0. Prerequisites

Run this notebook with a Jupyter kernel that has an **AMD ROCm PyTorch + vLLM** environment. A stock host Python won't work. Pick one:

### Option A (recommended): vLLM-ROCm container

The image's default entrypoint is `vllm serve`, so override it with `--entrypoint /bin/bash` to get a shell.

```bash
# on the host — mount wherever your models live to /models
docker run -it --rm --network host \
  --device /dev/kfd --device /dev/dri --group-add video \
  --security-opt seccomp=unconfined --ipc=host \
  -v /path/to/your/models:/models \
  --entrypoint /bin/bash \
  vllm/vllm-openai-rocm:v0.23.0

# inside the container
pip install jupyterlab
jupyter lab --ip=0.0.0.0 --port=8888 --allow-root --no-browser
```

Open the printed `http://127.0.0.1:8888/?token=…` URL in your browser to run the notebook there, or point your notebook client at that server as an existing/remote kernel.

### Option B: bare-metal

```bash
export PATH=/opt/rocm/bin:$PATH
python3 -m venv .venv && source .venv/bin/activate
pip install --upgrade pip ipykernel jupyterlab
pip install torch torchvision --index-url https://download.pytorch.org/whl/rocm<ver>
pip install vllm --extra-index-url <vLLM ROCm wheel index matching your ROCm>
python -m ipykernel install --user --name lmcache --display-name "Python (LMCache ROCm)"
```

Then start Jupyter (`jupyter lab`) and select the **Python (LMCache ROCm)** kernel.

### Model

Set `<MODEL>` in cells 2, 4, 5, 7 (and `MODEL` in the next cell) to a model that exists on the kernel's filesystem, keeping `--served-model-name` / `--tokenizer` consistent with it.

Run the next cell to check your setup.

In [ ]:
# Check the kernel has what the rest of the notebook needs.
import shutil, os, importlib

ok = True
try:
    import torch
    print(f"torch {torch.__version__} | HIP {torch.version.hip}")
    if not torch.version.hip:
        print("  !! not a ROCm/HIP torch"); ok = False
except Exception as e:
    print("torch: MISSING —", e); ok = False

try:
    vllm = importlib.import_module("vllm")
    print("vllm", getattr(vllm, "__version__", "?"))
except Exception as e:
    print("vllm: MISSING —", e); ok = False

print("vllm CLI:", shutil.which("vllm") or "NOT FOUND")

MODEL = "/models/gemma-4-31B-it"   # <-- set to your model dir (match cells 2, 4, 5, 7)
print(f"model {MODEL}:", "OK" if os.path.isdir(MODEL) else "MISSING")

print("\n==> ready" if ok else "\n==> environment incomplete — see section 0")

## 1. Install LMCache — built from source with HIP kernels

**On ROCm you must build LMCache from source**, not `pip install lmcache`. The PyPI wheel is CUDA-only: its native transfer kernels (`c_ops`) need `libcudart` and silently fall back to a slow Python path (~700 MB/s), which makes CPU offload/reload *slower* than recompute. `BUILD_WITH_HIP=1` compiles native HIP `c_ops` (fast D2H/H2D transfer) and makes `setup.py` auto-select `cupy-rocm`. The verify line must show `c_ops` loaded and `is_hip = True`. First build compiles kernels (~5-10 min).

In [ ]:
%%bash
set -e
# LMCache on ROCm must be built FROM SOURCE with HIP kernels. The PyPI wheel is
# CUDA-only: its native c_ops need libcudart and silently fall back to a slow
# Python KV-transfer path (~700 MB/s), making CPU offload/reload slower than
# recompute. BUILD_WITH_HIP=1 compiles native HIP c_ops and auto-selects cupy-rocm.
export PATH=/opt/rocm/bin:$PATH
python3 -c "import torch; assert torch.version.hip, 'need a ROCm/HIP torch'"

SRC=/workspace/LMCache
[ -d "$SRC/.git" ] || git clone --depth 1 https://github.com/LMCache/LMCache.git "$SRC"
cd "$SRC"
pip install -q -r requirements/build.txt

#   PYTORCH_ROCM_ARCH: gfx950 = MI350X/MI355X ; gfx942 = MI300X/MI325X
MAX_JOBS="${MAX_JOBS:-8}" PYTORCH_ROCM_ARCH="${PYTORCH_ROCM_ARCH:-gfx950}" \
  TORCH_DONT_CHECK_COMPILER_ABI=1 CXX=hipcc BUILD_WITH_HIP=1 \
  pip install --no-build-isolation . 2>&1 | tail -3
pip install -q "grpcio==1.78.0" 2>&1 | tail -1 || true   # source build can bump grpcio; keep vLLM's pin

echo "---- verify (native c_ops must load; is_hip must be True) ----"
python3 -c "import lmcache, cupy; from lmcache import c_ops; from cupy_backends.cuda.api import runtime as r; print('lmcache', lmcache.__version__, '| cupy', cupy.__version__, '| c_ops', c_ops.__file__.split('/')[-1], '| is_hip =', getattr(r,'is_hip',False))"

## 2. Baseline server — vLLM prefix cache only

A lower `--gpu-memory-utilization` (0.4) makes GPU memory the binding constraint so the working set in cell 4 overflows the KV pool. **Don't** pin the pool with `--kv-cache-memory-bytes` or a small `--max-model-len` — on a hybrid model that shrinks the sliding-window pool ~5×. Launches in the background; the next cell waits for it.

In [ ]:
%%bash
pkill -f "vllm serve" 2>/dev/null && sleep 4 || true
pkill -f "lmcache server" 2>/dev/null || true

# Prefix-cache-only baseline. Don't pin the KV pool small: on a hybrid model the
# sliding-window allocator would shrink it ~5x, and --max-model-len must stay 'auto'.
# Instead lower --gpu-memory-utilization so GPU memory is the binding constraint and
# the workload (cell 4) overflows it.
nohup vllm serve /models/gemma-4-31B-it \
  --served-model-name gemma-4-31B-it \
  --tensor-parallel-size 1 \
  --max-model-len auto \
  --gpu-memory-utilization 0.4 \
  --enable-prefix-caching \
  > /tmp/vllm.log 2>&1 &
echo "launching prefix-only server (log: /tmp/vllm.log); first boot ~4-6 min" 

## 3. Wait for the server (live log)

Polls `/v1/models` and prints the tail of the server log each attempt, so you can watch weight-load → compile → ready.

In [ ]:
import time, requests, subprocess
for i in range(80):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print("\n==> server ready"); break
    except requests.RequestException:
        pass
    tail = subprocess.run(["tail","-n","3","/tmp/vllm.log"], capture_output=True, text=True).stdout
    print(f"--- waiting {i+1}/80 ---\n{tail}", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("server did not start - see /tmp/vllm.log")

## 4. Benchmark the baseline (two passes)

`lmcache bench engine` runs the **`long-doc-qa`** workload. Two things make the reload win measurable: `--ignore-eos --ldqa-max-output-length 2048` makes it **decode-bound and deterministic** (a prefill-bound run hides the gap), and the working set is sized to **overflow the measured GPU pool** (read from vLLM's own startup log, so it's correct for hybrid attention). Run twice with a fixed `--seed`: PASS 1 (cold) primes, PASS 2 (warm) re-sends identical requests. Prefix-cache only + overflow ⇒ **PASS 2 ≈ PASS 1** (evicted KV recomputed).

In [ ]:
import subprocess, json, os, re

MODEL_NAME  = "gemma-4-31B-it"
DOC_LEN     = 24000   # tokens per long document
OUT_LEN     = 2048    # decode-bound (with --ignore-eos) so the reload win is visible
CONCURRENCY = 8       # >= hybrid batch, to saturate the pool
OVERFLOW    = 2.0     # working set = OVERFLOW x the GPU KV pool -> genuine spill to CPU

def pool_from_log(log: str = "/tmp/vllm.log") -> tuple[float, int]:
    """Read vLLM's own reported KV pool (GiB and tokens) from its startup log."""
    txt = open(log).read()
    gib = re.search(r"Available KV cache memory:\s*([0-9.]+)\s*GiB", txt)
    tok = re.search(r"GPU KV cache size:\s*([\d,]+)\s*tokens", txt)
    if not (gib and tok):
        raise RuntimeError("KV pool not found in vLLM log yet — is the server ready?")
    return float(gib.group(1)), int(tok.group(1).replace(",", ""))

def bench() -> float:
    """Run one long-doc-qa pass against the engine; return mean TTFT in seconds.

    Sizes the working set to OVERFLOW x the *measured* GPU pool so it spills on any
    GPU, and reads tokens/GB from vLLM's own report (correct for hybrid attention).
    """
    pool_gib, pool_tok = pool_from_log()
    tokens_per_gb = round(pool_tok / pool_gib)
    volume_gb     = round(OVERFLOW * pool_gib)
    out = "/tmp/bench"; os.makedirs(out, exist_ok=True)
    cmd = ["lmcache", "bench", "engine",
        "--engine-url", "http://localhost:8000", "--model", MODEL_NAME,
        "--workload", "long-doc-qa",
        "--kv-cache-volume", str(volume_gb), "--tokens-per-gb-kvcache", str(tokens_per_gb),
        "--ldqa-document-length", str(DOC_LEN), "--ldqa-query-per-document", "1",
        "--ldqa-num-inflight-requests", str(CONCURRENCY), "--ldqa-max-output-length", str(OUT_LEN),
        "--ignore-eos", "--no-interactive", "--quiet",
        "--json", "--no-csv", "--output-dir", out, "--seed", "555"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    summary = f"{out}/bench_summary.json"
    if not os.path.exists(summary):
        raise RuntimeError(f"bench produced no summary:\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return json.load(open(summary))["results"]["mean_ttft_ms"] / 1000.0

pg, pt = pool_from_log()
print(f"GPU KV pool: {pg:.1f} GiB ({pt:,} tokens) | working set ~{OVERFLOW}x -> overflow")
prefix_cold = bench(); print(f"prefix-only  PASS1 cold: {prefix_cold:6.2f} s")
prefix_warm = bench(); print(f"prefix-only  PASS2 warm: {prefix_warm:6.2f} s   ({prefix_cold/prefix_warm:.2f}x)")

## 5. Restart with LMCache — prefix cache + CPU KV offload

A separate `lmcache server` holds KV in CPU DRAM; vLLM talks to it via `LMCacheMPConnector` (`lmcache.mp.mq_timeout` is raised so registering the large pinned CPU pool doesn't time out). MP mode hashes chunks with **blake3** (deterministic), so the two processes agree on keys with no `PYTHONHASHSEED`. Same model, same `--gpu-memory-utilization` and `--max-model-len auto` as the baseline — only the CPU tier is added.

In [ ]:
%%bash
pkill -f "vllm serve" 2>/dev/null && sleep 4 || true
pkill -f "lmcache server" 2>/dev/null && sleep 2 || true

# CPU-DRAM KV cache server (start first so its pinned pool pre-expands).
nohup lmcache server \
  --host 127.0.0.1 --port 5555 \
  --l1-size-gb 400 --eviction-policy LRU --eviction-trigger-watermark 0.95 --chunk-size 256 \
  > /tmp/lmcache_server.log 2>&1 &
sleep 8

# vLLM + LMCache MP connector — SAME model / util / auto pool as the baseline.
nohup vllm serve /models/gemma-4-31B-it \
  --served-model-name gemma-4-31B-it \
  --tensor-parallel-size 1 \
  --max-model-len auto \
  --gpu-memory-utilization 0.4 \
  --enable-prefix-caching \
  --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both","kv_connector_extra_config":{"lmcache.mp.port":5555,"lmcache.mp.mq_timeout":900}}' \
  > /tmp/vllm.log 2>&1 &
echo "launching prefix+LMCache server; first boot ~4-6 min" 

## 6. Wait for the LMCache server

In [ ]:
for i in range(80):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print("\n==> server ready"); break
    except requests.RequestException:
        pass
    tail = subprocess.run(["tail","-n","3","/tmp/vllm.log"], capture_output=True, text=True).stdout
    print(f"--- waiting {i+1}/80 ---\n{tail}", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("server did not start - see /tmp/vllm.log")

## 7. Benchmark with LMCache (same two passes)

Identical workload. PASS 1 primes the CPU cache; **PASS 2 measures steady state** — evicted KV is reloaded from CPU instead of recomputed, so PASS 2 drops.

In [ ]:
lmcache_cold = bench(); print(f"LMCache  PASS1 cold: {lmcache_cold:6.2f} s")
lmcache_warm = bench(); print(f"LMCache  PASS2 warm: {lmcache_warm:6.2f} s   ({lmcache_cold/lmcache_warm:.2f}x)")

## 8. Proof it was the cache

The LMCache server logs `Stored` (cold) then `Retrieved` (warm). Only `Stored` and never `Retrieved` → cache keys mismatch (e.g. the two processes disagree on `--chunk-size` or `--hash-algorithm`).

In [ ]:
%%bash
grep -iE "Stored [0-9]+ tokens|Retrieved [0-9]+ tokens" /tmp/lmcache_server.log | tail -6

## 9. Side-by-side result

Cold bars are similar (both recompute on first pass). The **warm bars are the story**: prefix-only stays high (recomputes evicted KV), prefix+LMCache drops (reloads from CPU).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

groups = ["Cold (PASS 1)", "Warm (PASS 2)"]
x = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(6.5, 4))
b1 = ax.bar(x - w/2, [prefix_cold, prefix_warm], w, label="prefix cache only", color="#c0504d")
b2 = ax.bar(x + w/2, [lmcache_cold, lmcache_warm], w, label="prefix + LMCache", color="#4f81bd")
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("Mean TTFT (s)")
ax.set_title("Warm pass: LMCache reloads evicted KV instead of recomputing")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{b.get_height():.1f}s",
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print(f"Warm TTFT: prefix-only {prefix_warm:.1f}s vs LMCache {lmcache_warm:.1f}s "
      f"-> {prefix_warm/lmcache_warm:.1f}x faster")

## 10. When does LMCache help?

Only when **both** hold:

| Condition | Why |
|---|---|
| **Long context reused** | Reload only pays off if recompute (long prefill) is expensive. |
| **GPU HBM under pressure** | If the working set fits in GPU, vLLM's prefix cache already serves reuse and LMCache just adds transfer cost. |

That's why this demo overflows the GPU pool with long requests. With a single short request, or a large GPU pool, prefix-cache alone is enough and LMCache shows no benefit — that's expected.

### AMD / ROCm gotchas
- **Build LMCache from source with `BUILD_WITH_HIP=1`** — the PyPI wheel is CUDA-only; its `c_ops` fall back to a slow Python transfer path and LMCache ends up *slower* than recompute.
- **Don't pin the KV pool** (`--kv-cache-memory-bytes` / small `--max-model-len`) on hybrid models — it shrinks the sliding-window pool ~5×. Lower `--gpu-memory-utilization` instead.
- **`LMCacheMPConnector`** (separate `lmcache server`) — use on ROCm; the in-engine connector faults under concurrency. Raise `lmcache.mp.mq_timeout` for large CPU pools.
- **`PYTHONHASHSEED` not needed** — MP mode hashes with blake3 (deterministic). Only matters under `--hash-algorithm builtin`.